# Pooled Pilot Analysis (2026-03-09)

Pooled analysis of both condition-1 pilots: March 2 (n=8) and March 9 (n=29), total n=37. All subjects are condition 1 (item recognition). Subjects are renumbered sequentially (1–37), with `dataset` indicating origin (mar02 or mar09).

### Analyses

**Study phase (orienting).** 2(flanker gender compatibility: compatible/incompatible) × 3(flanker emotion: angry/happy/neutral) repeated-measures ANOVAs on accuracy and RT.

**Test phase (memory).** 4-level one-way repeated-measures ANOVAs on p(old) and RT, where the levels are angry-old, happy-old, neutral-old, and new. Follow-up comparisons use the omnibus error term.

**Supplementary.** Signal detection analysis (d', criterion) as in the individual pilot notebooks.

### Design recap

**Study phase.** On each trial, a neutral target face appears flanked on both sides by the same emotional face (angry, happy, or neutral). The participant judges the target's gender (Z=female, M=male). The display stays on screen for a fixed 3 seconds regardless of response. The factorial design crosses target gender (2) × flanker gender (2) × flanker emotion (3) = 12 trial types, each repeated 5 times, for 60 study trials per subject.

**Test phase (condition 1 — item recognition).** Each participant sees 120 single neutral faces (60 old + 60 new) and judges each as old or new. The old/new key mapping is counterbalanced across subjects via `key_mapping`.

In [1]:
import pandas as pd
from pathlib import Path
from statistics import NormalDist

z = NormalDist().inv_cdf  # equivalent to scipy.stats.norm.ppf

# Resolve CSV path relative to this notebook's location
_nb_dir = Path.cwd() if '__vsc_ipynb_file__' not in dir() else Path(__vsc_ipynb_file__).parent
df = pd.read_csv(_nb_dir / '2026_03_09_pooled_37subj.csv')
df['correct'] = df['correct'].astype('boolean')  # nullable bool

print(f'{len(df)} rows, {df.subject_number.nunique()} subjects')
print(f'Conditions: {df.groupby("condition").subject_number.nunique().to_dict()}')
print(f'Datasets: {df.groupby("dataset").subject_number.nunique().to_dict()}')

6660 rows, 37 subjects
Conditions: {1: 37}
Datasets: {'mar02': 8, 'mar09': 29}


## Exclusion Criteria

Exclude subjects with zeros in several cells of the study-phase design. The study phase has 12 cells (2 target gender × 2 flanker gender × 3 flanker emotion), each with 5 trials. Subjects with ≥6 zero-correct cells (out of 12) are excluded — these subjects were either pressing reversed keys or not performing the task.

In [2]:
ZERO_CELL_THRESHOLD = 6  # exclude if >= this many cells have 0 correct

study_all = df[df.phase == 'study']

# Count zero-correct cells per subject in the 12-cell design
cell_correct = study_all.groupby(
    ['subject_number', 'dataset', 'target_gender', 'flanker_gender', 'flanker_emotion']
).correct.sum().reset_index()
zero_cells = cell_correct.groupby(['subject_number', 'dataset']).apply(
    lambda g: (g.correct == 0).sum()
).reset_index(name='zero_cells')

# Also compute overall study accuracy
subj_study_acc = study_all.groupby('subject_number').correct.mean()
zero_cells['study_accuracy'] = zero_cells.subject_number.map(subj_study_acc)

print(f'Study-phase design: 12 cells per subject (5 trials each)')
print(f'Subjects with zero-correct cells:')
has_zeros = zero_cells[zero_cells.zero_cells > 0].sort_values('zero_cells', ascending=False)
if len(has_zeros) == 0:
    print('  None')
else:
    for _, row in has_zeros.iterrows():
        flag = ' ** EXCLUDED' if row.zero_cells >= ZERO_CELL_THRESHOLD else ''
        print(f'  Subject {int(row.subject_number)} ({row.dataset}): '
              f'{int(row.zero_cells)}/12 zero cells, {row.study_accuracy:.1%} accuracy{flag}')
print()

# Apply filter
excluded = zero_cells[zero_cells.zero_cells >= ZERO_CELL_THRESHOLD].subject_number.tolist()
keep = zero_cells[zero_cells.zero_cells < ZERO_CELL_THRESHOLD].subject_number.tolist()
df = df[df.subject_number.isin(keep)].copy()
print(f'{len(excluded)} excluded, {df.subject_number.nunique()} subjects remain')
print(f'Datasets: {df.groupby("dataset").subject_number.nunique().to_dict()}')

Study-phase design: 12 cells per subject (5 trials each)
Subjects with zero-correct cells:
  Subject 19 (mar09): 11/12 zero cells, 1.7% accuracy ** EXCLUDED
  Subject 21 (mar09): 11/12 zero cells, 1.7% accuracy ** EXCLUDED
  Subject 27 (mar09): 10/12 zero cells, 5.0% accuracy ** EXCLUDED
  Subject 7 (mar02): 9/12 zero cells, 5.0% accuracy ** EXCLUDED
  Subject 3 (mar02): 6/12 zero cells, 38.3% accuracy ** EXCLUDED
  Subject 14 (mar09): 5/12 zero cells, 50.0% accuracy
  Subject 22 (mar09): 5/12 zero cells, 48.3% accuracy
  Subject 25 (mar09): 4/12 zero cells, 51.7% accuracy
  Subject 8 (mar02): 1/12 zero cells, 63.3% accuracy

5 excluded, 32 subjects remain
Datasets: {'mar02': 6, 'mar09': 26}


### Exclusion summary

Five subjects are excluded for having ≥6 zero-correct cells out of 12 in the study-phase design. Four of these scored 1.7–5% overall accuracy (systematic key reversal). One scored 38.3% with 6 zero cells. Three additional subjects have 1–5 zero cells but are retained; their overall accuracy (48–63%) is low but they have data in most cells.

After exclusion: 32 subjects (6 mar02, 26 mar09).

## Study Phase (Orienting)

The study phase serves as the orienting task. Each trial presents a neutral target flanked by emotional faces. The participant judges the target's gender. We analyze accuracy and RT in a 2(flanker gender compatibility: compatible = flanker gender matches target, incompatible = mismatch) × 3(flanker emotion: angry, happy, neutral) repeated-measures design.

**Gender compatibility** tests whether flanker gender interferes with or facilitates the gender judgment. **Flanker emotion** tests whether the flanker's emotional expression modulates performance.

In [3]:
study = df[df.phase == 'study'].copy()
study['compatible'] = study.target_gender == study.flanker_gender
study['compat_label'] = study.compatible.map({True: 'compatible', False: 'incompatible'})

# Per-subject means in each cell of the 2x3 design
study_acc_subj = study.groupby(
    ['subject_number', 'compat_label', 'flanker_emotion']
).correct.mean().reset_index(name='accuracy')

study_rt_subj = study[~study.timed_out].groupby(
    ['subject_number', 'compat_label', 'flanker_emotion']
).rt.mean().reset_index(name='mean_rt')

# Group means: accuracy
acc_table = study_acc_subj.groupby(['compat_label', 'flanker_emotion']).accuracy.agg(
    ['mean', 'std']
).round(3)
print('Study-phase accuracy by compatibility x emotion:')
print(acc_table.to_string())
print()

# Marginal means for accuracy
print('Marginal means (accuracy):')
print(f'  Compatible:   {study_acc_subj[study_acc_subj.compat_label == "compatible"].accuracy.mean():.3f}')
print(f'  Incompatible: {study_acc_subj[study_acc_subj.compat_label == "incompatible"].accuracy.mean():.3f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {study_acc_subj[study_acc_subj.flanker_emotion == emo].accuracy.mean():.3f}')
print()

# Group means: RT
rt_table = study_rt_subj.groupby(['compat_label', 'flanker_emotion']).mean_rt.agg(
    ['mean', 'std']
).round(1)
print('Study-phase RT (ms) by compatibility x emotion:')
print(rt_table.to_string())
print()

# Marginal means for RT
print('Marginal means (RT):')
print(f'  Compatible:   {study_rt_subj[study_rt_subj.compat_label == "compatible"].mean_rt.mean():.1f}')
print(f'  Incompatible: {study_rt_subj[study_rt_subj.compat_label == "incompatible"].mean_rt.mean():.1f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {study_rt_subj[study_rt_subj.flanker_emotion == emo].mean_rt.mean():.1f}')

Study-phase accuracy by compatibility x emotion:
                               mean    std
compat_label flanker_emotion              
compatible   angry            0.956  0.067
             happy            0.956  0.076
             neutral          0.928  0.135
incompatible angry            0.816  0.313
             happy            0.809   0.34
             neutral          0.881  0.247

Marginal means (accuracy):
  Compatible:   0.947
  Incompatible: 0.835
         angry: 0.886
         happy: 0.883
       neutral: 0.905

Study-phase RT (ms) by compatibility x emotion:
                                mean    std
compat_label flanker_emotion               
compatible   angry             999.7  292.8
             happy            1008.4  329.1
             neutral          1003.8  316.6
incompatible angry            1052.4  339.4
             happy            1044.7  352.6
             neutral          1028.6  351.2

Marginal means (RT):
  Compatible:   1003.9
  Incompatible: 1041.9


In [4]:
import math

# --- Exact p-values via incomplete beta function (no scipy needed) ---

def _betacf(a, b, x):
    """Continued fraction for regularized incomplete beta (Numerical Recipes)."""
    MAXIT, EPS = 200, 3e-12
    qab, qap, qam = a + b, a + 1.0, a - 1.0
    c = 1.0
    d = 1.0 / (1.0 - qab * x / qap) if abs(1.0 - qab * x / qap) > 1e-30 else 1.0 / 1e-30
    h = d
    for m in range(1, MAXIT + 1):
        m2 = 2 * m
        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-30: d = 1e-30
        c = 1.0 + aa / c
        if abs(c) < 1e-30: c = 1e-30
        d = 1.0 / d
        h *= d * c
        aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-30: d = 1e-30
        c = 1.0 + aa / c
        if abs(c) < 1e-30: c = 1e-30
        d = 1.0 / d
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < EPS:
            break
    return h

def _betai(a, b, x):
    """Regularized incomplete beta function I_x(a, b)."""
    if x <= 0: return 0.0
    if x >= 1: return 1.0
    lbeta = math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)
    front = math.exp(a * math.log(x) + b * math.log(1 - x) - lbeta)
    if x < (a + 1) / (a + b + 2):
        return front * _betacf(a, b, x) / a
    else:
        return 1.0 - front * _betacf(b, a, 1 - x) / b

def t_p_twotail(t_val, df):
    """Exact two-tailed p-value for t-distribution."""
    x = df / (df + t_val ** 2)
    return _betai(df / 2.0, 0.5, x)

def f_p(f_val, df1, df2):
    """Upper-tail p-value for F-distribution."""
    if f_val <= 0: return 1.0
    x = df2 / (df2 + df1 * f_val)
    return _betai(df2 / 2.0, df1 / 2.0, x)


# --- One-way repeated-measures ANOVA ---

def rm_anova_oneway(groups):
    """One-way RM ANOVA. Returns F, df1, df2, p, partial_eta_sq, ms_error."""
    k = len(groups)
    n = len(groups[0])
    grand = sum(sum(g) for g in groups) / (k * n)
    subj_m = [sum(groups[j][i] for j in range(k)) / k for i in range(n)]
    cond_m = [sum(g) / n for g in groups]
    ss_cond = n * sum((m - grand) ** 2 for m in cond_m)
    ss_subj = k * sum((m - grand) ** 2 for m in subj_m)
    ss_total = sum((groups[j][i] - grand) ** 2 for j in range(k) for i in range(n))
    ss_err = ss_total - ss_cond - ss_subj
    df1 = k - 1
    df2 = (k - 1) * (n - 1)
    ms_err = ss_err / df2 if df2 > 0 else float('nan')
    f_val = (ss_cond / df1) / ms_err if ss_err > 0 else float('nan')
    p = f_p(f_val, df1, df2)
    eta = ss_cond / (ss_cond + ss_err)
    return f_val, df1, df2, p, eta, ms_err


# --- Two-way repeated-measures ANOVA ---

def rm_anova_twoway(data, a_levels, b_levels):
    """
    Two-way repeated-measures ANOVA.

    data: dict mapping (a_level, b_level) -> list of n values (one per subject)
    a_levels: list of levels for factor A
    b_levels: list of levels for factor B

    Returns dict with keys 'A', 'B', 'AxB', each containing:
        F, df1, df2, p, partial_eta_sq, ms_error
    """
    a = len(a_levels)
    b = len(b_levels)
    n = len(data[(a_levels[0], b_levels[0])])

    # Build 3D array: Y[i][j][k] = subject i, A level j, B level k
    Y = [[[data[(a_levels[j], b_levels[k])][i]
           for k in range(b)] for j in range(a)] for i in range(n)]

    # Grand mean
    gm = sum(Y[i][j][k] for i in range(n) for j in range(a) for k in range(b)) / (n * a * b)

    # Marginal means
    subj_m = [sum(Y[i][j][k] for j in range(a) for k in range(b)) / (a * b) for i in range(n)]
    a_m = [sum(Y[i][j][k] for i in range(n) for k in range(b)) / (n * b) for j in range(a)]
    b_m = [sum(Y[i][j][k] for i in range(n) for j in range(a)) / (n * a) for k in range(b)]
    ab_m = [[sum(Y[i][j][k] for i in range(n)) / n for k in range(b)] for j in range(a)]
    sa_m = [[sum(Y[i][j][k] for k in range(b)) / b for j in range(a)] for i in range(n)]
    sb_m = [[sum(Y[i][j][k] for j in range(a)) / a for k in range(b)] for i in range(n)]

    # Sums of squares
    ss_a = n * b * sum((a_m[j] - gm) ** 2 for j in range(a))
    ss_b = n * a * sum((b_m[k] - gm) ** 2 for k in range(b))
    ss_ab = n * sum((ab_m[j][k] - a_m[j] - b_m[k] + gm) ** 2
                    for j in range(a) for k in range(b))
    ss_s = a * b * sum((subj_m[i] - gm) ** 2 for i in range(n))
    ss_as = b * sum((sa_m[i][j] - a_m[j] - subj_m[i] + gm) ** 2
                    for i in range(n) for j in range(a))
    ss_bs = a * sum((sb_m[i][k] - b_m[k] - subj_m[i] + gm) ** 2
                    for i in range(n) for k in range(b))
    ss_total = sum((Y[i][j][k] - gm) ** 2
                   for i in range(n) for j in range(a) for k in range(b))
    ss_abs = ss_total - ss_a - ss_b - ss_ab - ss_s - ss_as - ss_bs

    # Degrees of freedom
    df_a = a - 1
    df_b = b - 1
    df_ab = df_a * df_b
    df_as = df_a * (n - 1)
    df_bs = df_b * (n - 1)
    df_abs = df_ab * (n - 1)

    results = {}
    for label, ss_eff, df_eff, ss_e, df_e in [
        ('A', ss_a, df_a, ss_as, df_as),
        ('B', ss_b, df_b, ss_bs, df_bs),
        ('AxB', ss_ab, df_ab, ss_abs, df_abs),
    ]:
        ms_eff = ss_eff / df_eff if df_eff > 0 else 0
        ms_e = ss_e / df_e if df_e > 0 else float('nan')
        f_val = ms_eff / ms_e if ms_e > 0 else float('nan')
        p = f_p(f_val, df_eff, df_e)
        eta = ss_eff / (ss_eff + ss_e) if (ss_eff + ss_e) > 0 else 0
        results[label] = {
            'F': f_val, 'df1': df_eff, 'df2': df_e,
            'p': p, 'eta_sq': eta, 'ms_error': ms_e
        }

    return results


# --- Build data dicts for the 2x3 study-phase ANOVAs ---

subjects = sorted(study_acc_subj.subject_number.unique())
n_subj = len(subjects)
a_levels = ['compatible', 'incompatible']
b_levels = ['angry', 'happy', 'neutral']

# Accuracy
acc_data = {}
for al in a_levels:
    for bl in b_levels:
        mask = (study_acc_subj.compat_label == al) & (study_acc_subj.flanker_emotion == bl)
        vals = study_acc_subj[mask].set_index('subject_number').loc[subjects, 'accuracy'].tolist()
        acc_data[(al, bl)] = vals

print(f'2(compatibility) x 3(emotion) RM ANOVA on accuracy (n={n_subj}):')
print()
acc_results = rm_anova_twoway(acc_data, a_levels, b_levels)
for label, name in [('A', 'Compatibility'), ('B', 'Emotion'), ('AxB', 'Compatibility x Emotion')]:
    r = acc_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")
print()

# RT
rt_data = {}
for al in a_levels:
    for bl in b_levels:
        mask = (study_rt_subj.compat_label == al) & (study_rt_subj.flanker_emotion == bl)
        vals = study_rt_subj[mask].set_index('subject_number').loc[subjects, 'mean_rt'].tolist()
        rt_data[(al, bl)] = vals

print(f'2(compatibility) x 3(emotion) RM ANOVA on RT (n={n_subj}):')
print()
rt_results = rm_anova_twoway(rt_data, a_levels, b_levels)
for label, name in [('A', 'Compatibility'), ('B', 'Emotion'), ('AxB', 'Compatibility x Emotion')]:
    r = rt_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")

2(compatibility) x 3(emotion) RM ANOVA on accuracy (n=32):

  Compatibility: F(1,31) = 5.113, p = 0.031, partial eta^2 = 0.142
  Emotion: F(2,62) = 2.067, p = 0.135, partial eta^2 = 0.062
  Compatibility x Emotion: F(2,62) = 7.923, p = 0.001, partial eta^2 = 0.204

2(compatibility) x 3(emotion) RM ANOVA on RT (n=32):

  Compatibility: F(1,31) = 3.531, p = 0.070, partial eta^2 = 0.102
  Emotion: F(2,62) = 0.212, p = 0.810, partial eta^2 = 0.007
  Compatibility x Emotion: F(2,62) = 0.170, p = 0.844, partial eta^2 = 0.005


### Study phase interpretation

**Gender compatibility affects orienting accuracy.** The compatibility main effect is significant: F(1,31) = 5.11, p = .031, partial eta-sq = .142. Compatible trials (flanker gender = target gender) yield 94.7% accuracy vs 83.5% for incompatible trials. The RT effect is in the same direction (1004 vs 1042 ms) but only marginal, F(1,31) = 3.53, p = .070.

**Compatibility x emotion interaction is highly significant in accuracy.** F(2,62) = 7.92, p = .001, partial eta-sq = .204. The pattern: on compatible trials, accuracy is near ceiling regardless of emotion (~93–96%). On incompatible trials, emotional flankers (angry 81.6%, happy 80.9%) cause substantially more interference than neutral flankers (88.1%). This suggests that emotional expressions amplify the gender-compatibility interference effect — when the flanker gender mismatches the target, an emotional expression makes it even harder to judge the target's gender.

**No interaction in RT.** The compatibility x emotion interaction is absent in RT, F(2,62) = 0.17, p = .844. The accuracy interaction may reflect a speed–accuracy tradeoff where subjects maintain similar RTs but make more errors on emotionally incompatible trials.

## Test Phase (Memory)

Memory is analyzed as probability of saying "old" — p(old) — in a 4-level one-way design: angry-old, happy-old, neutral-old, and new. For old items, p(old) is the hit rate; for new items, p(old) is the false alarm rate. Memory is above chance when p(old) for old items exceeds p(old) for new items.

RT is analyzed in the same 4-level design. Follow-up comparisons use the MSE from the omnibus ANOVA.

In [5]:
test = df[df.phase == 'test'].copy()

# Create a 4-level condition variable
# For old items: use study_flanker_emotion (angry, happy, neutral)
# For new items: label as 'new'
test['memory_cond'] = test.apply(
    lambda r: r.study_flanker_emotion if r.stimulus_type == 'old' else 'new', axis=1
)

# p(old) = probability of responding "old"
# For old items, correct=True means they said "old" (hit)
# For new items, correct=False means they said "old" (false alarm)
test['said_old'] = test.apply(
    lambda r: bool(r.correct) if r.stimulus_type == 'old' else not bool(r.correct), axis=1
)

# Per-subject p(old) in each condition
pold_subj = test.groupby(['subject_number', 'memory_cond']).said_old.mean().reset_index(name='p_old')

# Per-subject RT in each condition (non-timed-out only)
rt_test_subj = test[~test.timed_out].groupby(
    ['subject_number', 'memory_cond']
).rt.mean().reset_index(name='mean_rt')

# Group means
cond_order = ['angry', 'happy', 'neutral', 'new']

print('Test-phase p(old) by condition:')
for cond in cond_order:
    vals = pold_subj[pold_subj.memory_cond == cond].p_old
    label = f'{cond}-old' if cond != 'new' else 'new'
    print(f'  {label:>12}: M={vals.mean():.3f}, SD={vals.std():.3f}')
print()

print('Test-phase RT (ms) by condition:')
for cond in cond_order:
    vals = rt_test_subj[rt_test_subj.memory_cond == cond].mean_rt
    label = f'{cond}-old' if cond != 'new' else 'new'
    print(f'  {label:>12}: M={vals.mean():.1f}, SD={vals.std():.1f}')

Test-phase p(old) by condition:
     angry-old: M=0.478, SD=0.143
     happy-old: M=0.463, SD=0.158
   neutral-old: M=0.456, SD=0.146
           new: M=0.311, SD=0.145

Test-phase RT (ms) by condition:
     angry-old: M=1043.7, SD=288.8
     happy-old: M=1089.0, SD=278.6
   neutral-old: M=1073.6, SD=283.4
           new: M=1072.7, SD=282.6


In [6]:
# --- Helper: follow-up comparisons using omnibus MSE ---

def anova_followup(means_a, means_b, ms_error, df_error, label_a, label_b):
    """Pairwise comparison using omnibus error term.
    means_a, means_b: per-subject values (lists of length n).
    ms_error: MSE from the omnibus ANOVA.
    df_error: df_error from the omnibus ANOVA.
    Returns t, df, p (two-tailed), mean_diff.
    """
    n = len(means_a)
    diff = sum(a - b for a, b in zip(means_a, means_b)) / n
    se = math.sqrt(2 * ms_error / n)
    if se == 0:
        return 0.0, df_error, 1.0, diff
    t_val = diff / se
    p = t_p_twotail(t_val, df_error)
    return t_val, df_error, p, diff


# --- 4-level one-way RM ANOVAs ---

test_subjects = sorted(pold_subj.subject_number.unique())
n_test = len(test_subjects)

# Build groups for p(old)
pold_groups = []
for cond in cond_order:
    vals = pold_subj[pold_subj.memory_cond == cond].set_index('subject_number').loc[test_subjects, 'p_old'].tolist()
    pold_groups.append(vals)

f_val_po, df1_po, df2_po, p_po, eta_po, mse_po = rm_anova_oneway(pold_groups)
print(f"One-way RM ANOVA on p(old), 4 levels (n={n_test}):")
print(f"  F({df1_po},{df2_po}) = {f_val_po:.3f}, p = {p_po:.3f}, partial eta^2 = {eta_po:.3f}")
print(f"  MSE = {mse_po:.6f}")
print()

# Build groups for RT
rt_groups = []
for cond in cond_order:
    vals = rt_test_subj[rt_test_subj.memory_cond == cond].set_index('subject_number').loc[test_subjects, 'mean_rt'].tolist()
    rt_groups.append(vals)

f_val_rt, df1_rt, df2_rt, p_rt, eta_rt, mse_rt = rm_anova_oneway(rt_groups)
print(f"One-way RM ANOVA on RT, 4 levels (n={n_test}):")
print(f"  F({df1_rt},{df2_rt}) = {f_val_rt:.3f}, p = {p_rt:.3f}, partial eta^2 = {eta_rt:.3f}")
print(f"  MSE = {mse_rt:.1f}")
print()

# --- Follow-up comparisons using omnibus MSE ---

print('Follow-up comparisons on p(old) (using omnibus MSE):')
print()
print('  Each emotion vs new (is memory above chance?):')
for i, emo in enumerate(['angry', 'happy', 'neutral']):
    t, dfe, p, md = anova_followup(pold_groups[i], pold_groups[3], mse_po, df2_po,
                                    f'{emo}-old', 'new')
    print(f'    {emo}-old vs new: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()
print('  Pairwise among old emotions:')
emo_pairs = [('angry', 'happy', 0, 1), ('angry', 'neutral', 0, 2), ('happy', 'neutral', 1, 2)]
for e1, e2, i1, i2 in emo_pairs:
    t, dfe, p, md = anova_followup(pold_groups[i1], pold_groups[i2], mse_po, df2_po, e1, e2)
    print(f'    {e1}-old vs {e2}-old: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

print('Follow-up comparisons on RT (using omnibus MSE):')
print()
print('  Each emotion vs new:')
for i, emo in enumerate(['angry', 'happy', 'neutral']):
    t, dfe, p, md = anova_followup(rt_groups[i], rt_groups[3], mse_rt, df2_rt,
                                    f'{emo}-old', 'new')
    print(f'    {emo}-old vs new: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.1f} ms')
print()
print('  Pairwise among old emotions:')
for e1, e2, i1, i2 in emo_pairs:
    t, dfe, p, md = anova_followup(rt_groups[i1], rt_groups[i2], mse_rt, df2_rt, e1, e2)
    print(f'    {e1}-old vs {e2}-old: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.1f} ms')

One-way RM ANOVA on p(old), 4 levels (n=32):
  F(3,93) = 14.015, p = 0.000, partial eta^2 = 0.311
  MSE = 0.013852

One-way RM ANOVA on RT, 4 levels (n=32):
  F(3,93) = 1.587, p = 0.198, partial eta^2 = 0.049
  MSE = 7224.2

Follow-up comparisons on p(old) (using omnibus MSE):

  Each emotion vs new (is memory above chance?):
    angry-old vs new: t(93) = 5.682, p = 0.000, diff = 0.167
    happy-old vs new: t(93) = 5.151, p = 0.000, diff = 0.152
    neutral-old vs new: t(93) = 4.939, p = 0.000, diff = 0.145

  Pairwise among old emotions:
    angry-old vs happy-old: t(93) = 0.531, p = 0.597, diff = 0.016
    angry-old vs neutral-old: t(93) = 0.743, p = 0.459, diff = 0.022
    happy-old vs neutral-old: t(93) = 0.212, p = 0.832, diff = 0.006

Follow-up comparisons on RT (using omnibus MSE):

  Each emotion vs new:
    angry-old vs new: t(93) = -1.365, p = 0.175, diff = -29.0 ms
    happy-old vs new: t(93) = 0.769, p = 0.444, diff = 16.3 ms
    neutral-old vs new: t(93) = 0.044, p = 0.965

### Test phase interpretation

**Memory is reliably above chance.** The 4-level p(old) ANOVA is highly significant: F(3,93) = 14.02, p < .001, partial eta-sq = .311. This is driven by the old–new difference: all three old-emotion conditions produce higher p(old) than new items (all t > 4.9, all p < .001). Angry-old (.478), happy-old (.463), and neutral-old (.456) all exceed the false alarm rate (.311) by .14–.17.

**No emotion modulation of p(old).** Pairwise comparisons among the three old-emotion conditions are all far from significance (all p > .45, all diffs < .022). The hit rate is essentially flat across emotions.

**Angry-flanker RT advantage.** The omnibus RT ANOVA is not significant overall, F(3,93) = 1.59, p = .198. However, the angry-old vs happy-old comparison reaches significance: t(93) = −2.13, p = .035, diff = −45.3 ms. Subjects respond faster to items originally studied with angry flankers (1044 ms) than happy flankers (1089 ms). Angry-old is also numerically faster than new items (1044 vs 1073 ms, p = .175) and neutral-old (1044 vs 1074 ms, p = .162), though neither reaches significance. This pattern is consistent with Gordon's observation of an angry-flanker advantage that is "more apparent in RT than in accuracy."

**Interpreting the RT effect.** The faster RT for angry-studied items, in the absence of an accuracy difference, could reflect enhanced memory fluency: items encoded with angry flankers are recognized more quickly, even though the probability of calling them "old" is similar across emotions. This is a weak effect given the nonsignificant omnibus, but the direction is consistent with emotional enhancement of memory trace strength.

## Supplementary: Signal Detection Analysis

d' (sensitivity) and criterion c computed per subject per flanker emotion. False alarm rate is pooled across all new items. Edge correction follows Macmillan & Kaplan (1985).

In [7]:
def edge_correct(rate, n):
    """Apply Macmillan & Kaplan (1985) edge correction."""
    if rate == 0:
        return 0.5 / n
    if rate == 1:
        return 1 - 0.5 / n
    return rate


c1_test = df[(df.condition == 1) & (df.phase == 'test')].copy()
n_subj_c1 = c1_test.subject_number.nunique()

# Per-subject SDT metrics
sdt_per_subj = []
for subj, sdata in c1_test.groupby('subject_number'):
    new_items = sdata[sdata.stimulus_type == 'new']
    fa_rate_raw = (~new_items.correct).mean()
    fa_n = len(new_items)
    fa_rate = edge_correct(fa_rate_raw, fa_n)

    old_items = sdata[sdata.stimulus_type == 'old']
    for emotion, grp in old_items.groupby('study_flanker_emotion'):
        n = len(grp)
        hit_rate_raw = grp.correct.mean()
        hit_rate = edge_correct(hit_rate_raw, n)
        dprime = z(hit_rate) - z(fa_rate)
        criterion = -0.5 * (z(hit_rate) + z(fa_rate))
        sdt_per_subj.append({
            'subject': subj,
            'emotion': emotion,
            'hit_rate': hit_rate_raw,
            'fa_rate': fa_rate_raw,
            'd_prime': dprime,
            'criterion': criterion
        })

sdt_df = pd.DataFrame(sdt_per_subj)

# Group summary
sdt_summary = sdt_df.groupby('emotion').agg(
    N=('subject', 'count'),
    hit_rate_M=('hit_rate', 'mean'),
    hit_rate_SD=('hit_rate', 'std'),
    fa_rate_M=('fa_rate', 'mean'),
    d_prime_M=('d_prime', 'mean'),
    d_prime_SD=('d_prime', 'std'),
    criterion_M=('criterion', 'mean'),
    criterion_SD=('criterion', 'std'),
).round(3)

print(f'SDT Analysis (n={n_subj_c1} subjects)')
print(f'Mean FA rate: {sdt_df.fa_rate.mean():.3f} (SD={sdt_df.groupby("subject").fa_rate.first().std():.3f})')
print()
print(sdt_summary.to_string())
print()

# One-way RM ANOVA on d'
dprime_wide = sdt_df.pivot(index='subject', columns='emotion', values='d_prime')
angry_d = dprime_wide['angry'].tolist()
happy_d = dprime_wide['happy'].tolist()
neutral_d = dprime_wide['neutral'].tolist()

f_val_d, df1_d, df2_d, p_d, eta_d, mse_d = rm_anova_oneway([angry_d, happy_d, neutral_d])
print(f"One-way RM ANOVA on d' (flanker emotion):")
print(f"  F({df1_d},{df2_d}) = {f_val_d:.3f}, p = {p_d:.3f}, partial eta^2 = {eta_d:.3f}")
print()

# Follow-ups using omnibus MSE
print("Follow-up comparisons on d' (using omnibus MSE):")
d_groups = [angry_d, happy_d, neutral_d]
d_labels = ['angry', 'happy', 'neutral']
for i in range(3):
    for j in range(i + 1, 3):
        t, dfe, p, md = anova_followup(d_groups[i], d_groups[j], mse_d, df2_d,
                                        d_labels[i], d_labels[j])
        print(f'  {d_labels[i]} vs {d_labels[j]}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')

SDT Analysis (n=32 subjects)
Mean FA rate: 0.311 (SD=0.145)

          N  hit_rate_M  hit_rate_SD  fa_rate_M  d_prime_M  d_prime_SD  criterion_M  criterion_SD
emotion                                                                                          
angry    32       0.478        0.143      0.311      0.470       0.471        0.299         0.339
happy    32       0.462        0.158      0.311      0.440       0.441        0.314         0.374
neutral  32       0.456        0.146      0.311      0.402       0.503        0.333         0.357

One-way RM ANOVA on d' (flanker emotion):
  F(2,62) = 0.298, p = 0.744, partial eta^2 = 0.010

Follow-up comparisons on d' (using omnibus MSE):
  angry vs happy: t(62) = 0.335, p = 0.738, diff = 0.030
  angry vs neutral: t(62) = 0.770, p = 0.444, diff = 0.068
  happy vs neutral: t(62) = 0.434, p = 0.666, diff = 0.039


### SDT interpretation

The d' analysis converges with the p(old) results. Mean d' is weak but positive (angry = 0.47, happy = 0.44, neutral = 0.40), confirming above-chance recognition. The one-way ANOVA on d' is null: F(2,62) = 0.30, p = .744, partial eta-sq = .010. No pairwise comparisons approach significance. The d' ordering (angry > happy > neutral) is numerically consistent with the p(old) ordering but the effect size is tiny (eta-sq = .01).

The FA rate is .311 (SD = .145), stable across all pilots (~.31–.33). Subjects adopt a slightly conservative criterion (c = .30–.33), tending to say "new" more than "old."

## Summary

In [8]:
n_total = 37
n_excluded = n_total - df.subject_number.nunique()
n_kept = df.subject_number.nunique()

print(f'=== Pooled Pilot Summary (2026-03-09) ===')
print(f'{n_total} subjects pooled (8 mar02 + 29 mar09), {n_excluded} excluded (>=6 zero cells), {n_kept} analyzed')
print()

print('Study phase (orienting):')
print(f'  Overall accuracy: {study.correct.mean():.1%}')
print(f'  Mean RT: {study.loc[~study.timed_out, "rt"].mean():.0f} ms')
r = acc_results['A']
print(f"  Compatibility: F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}")
r = acc_results['B']
print(f"  Emotion:       F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}")
r = acc_results['AxB']
print(f"  Interaction:   F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}")
print()

print('Test phase (memory):')
print(f"  p(old) ANOVA: F({df1_po},{df2_po}) = {f_val_po:.3f}, p = {p_po:.3f}, eta^2 = {eta_po:.3f}")
print(f"  RT ANOVA:     F({df1_rt},{df2_rt}) = {f_val_rt:.3f}, p = {p_rt:.3f}, eta^2 = {eta_rt:.3f}")
print()

print('Supplementary SDT:')
for emotion, row in sdt_summary.iterrows():
    print(f"  {emotion:>7}: d'={row['d_prime_M']:.2f} (SD={row['d_prime_SD']:.2f}), "
          f"c={row['criterion_M']:.2f}, hit={row['hit_rate_M']:.2f}")
print(f"  d' ANOVA: F({df1_d},{df2_d}) = {f_val_d:.3f}, p = {p_d:.3f}, eta^2 = {eta_d:.3f}")

=== Pooled Pilot Summary (2026-03-09) ===
37 subjects pooled (8 mar02 + 29 mar09), 5 excluded (>=6 zero cells), 32 analyzed

Study phase (orienting):
  Overall accuracy: 89.1%
  Mean RT: 1017 ms
  Compatibility: F(1,31) = 5.113, p = 0.031
  Emotion:       F(2,62) = 2.067, p = 0.135
  Interaction:   F(2,62) = 7.923, p = 0.001

Test phase (memory):
  p(old) ANOVA: F(3,93) = 14.015, p = 0.000, eta^2 = 0.311
  RT ANOVA:     F(3,93) = 1.587, p = 0.198, eta^2 = 0.049

Supplementary SDT:
    angry: d'=0.47 (SD=0.47), c=0.30, hit=0.48
    happy: d'=0.44 (SD=0.44), c=0.31, hit=0.46
  neutral: d'=0.40 (SD=0.50), c=0.33, hit=0.46
  d' ANOVA: F(2,62) = 0.298, p = 0.744, eta^2 = 0.010
